# Notebook 9: Qwen2.5 with GTF-Penalty Decoding

Notebook này minh họa cách áp dụng GTF-Penalty để khắc phục lỗi lặp từ của Qwen2.5 tại thời điểm suy diễn (Inference) mà KHÔNG cần huấn luyện lại.

Yêu cầu: GPU (Khuyến nghị sử dụng Kaggle T4 x2 hoặc P100).

In [ ]:
!pip install -q transformers datasets peft bitsandbytes accelerate evaluate rouge_score bert_score "torchao>=0.16.0"

## 1. Import và Class GTF-Penalty

In [ ]:
import torch
import math
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, LogitsProcessor, LogitsProcessorList
from peft import PeftModel

class GTFPenaltyLogitsProcessor(LogitsProcessor):
    def __init__(self, vocab_frequencies: dict, prompt_length: int, alpha: float = 1.0, base_penalty: float = 0.6):
        self.vocab_frequencies = vocab_frequencies
        self.prompt_length = prompt_length
        self.alpha = alpha
        self.base_penalty = base_penalty
        
        if vocab_frequencies:
            self.c_max = max(vocab_frequencies.values())
        else:
            self.c_max = 1
            
        self.log_c_max = math.log(self.c_max) if self.c_max > 1 else 1.0

        max_token_id = max(vocab_frequencies.keys()) if vocab_frequencies else 0
        self.P_v = torch.zeros(max_token_id + 1)
        
        for token_id, count in vocab_frequencies.items():
            if count > 0:
                p_v = self.alpha * (1.0 - (math.log(count) / self.log_c_max))
            else:
                p_v = self.alpha
            self.P_v[token_id] = p_v
            
        self.P_v_device = None

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        batch_size, vocab_size = scores.shape
        
        if self.P_v_device is None or self.P_v_device.device != scores.device:
            if vocab_size > len(self.P_v):
                extended_P_v = torch.full((vocab_size,), self.alpha)
                extended_P_v[:len(self.P_v)] = self.P_v
                self.P_v = extended_P_v
            self.P_v_device = self.P_v.to(scores.device)

        for i in range(batch_size):
            # CHỈ ĐẾM CÁC TOKEN TRONG PHẦN GENERATED SUMMARY (BỎ QUA PROMPT)
            seq = input_ids[i][self.prompt_length:]
            unique_tokens, counts = torch.unique(seq, return_counts=True)
            
            valid_mask = unique_tokens < vocab_size
            unique_tokens = unique_tokens[valid_mask]
            counts = counts[valid_mask].float()
            
            if len(unique_tokens) > 0:
                penalty = (self.P_v_device[unique_tokens] * counts) + self.base_penalty
                scores[i, unique_tokens] -= penalty

        return scores


## 2. Load Model và Tokenizer

In [ ]:
import os
MODEL_NAME = 'Qwen/Qwen2.5-0.5B'
LORA_PATH = './qwen-lora-vietnews'

# Tự động tìm đường dẫn LoRA trên Kaggle
if os.path.exists('/kaggle/input'):
    found = False
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'adapter_config.json' in files:
            LORA_PATH = root
            print(f'Đã tìm thấy LoRA weights tại: {LORA_PATH}')
            found = True
            break
    if not found:
        print('CẢNH BÁO: Không tìm thấy LoRA weights trong /kaggle/input. Vui lòng Add Data -> Notebook Output!')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Loading base model...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True
)

print('Loading LoRA weights...')
try:
    model = PeftModel.from_pretrained(base_model, LORA_PATH)
    model = model.merge_and_unload() # Merge để sinh text nhanh hơn
    print('Loaded and merged LoRA weights successfully.')
except Exception as e:
    print(f'LỖI: Không thể load LoRA ({e}). Sẽ sử dụng Base Model.')
    model = base_model

model.eval()


## 3. Tạo Từ điển Tần suất (Vocab Frequencies)
Đọc tập train để đếm số lần xuất hiện của các token (C_v^{train}).

In [ ]:
from collections import Counter

print("Building vocab frequencies from train set...")
try:
    train_df = pd.read_csv("train_10k.csv")
    # Nối tất cả text để token hóa (hoặc lấy một mẫu nhỏ để đếm cho nhanh)
    sample_texts = train_df['article'].dropna().tolist()
    
    # Đếm tần suất - token hóa theo batch
    vocab_freq_counter = Counter()
    batch_size = 100
    for i in tqdm(range(0, len(sample_texts), batch_size), desc="Counting tokens"):
        batch = sample_texts[i:i+batch_size]
        inputs = tokenizer(batch, add_special_tokens=False)['input_ids']
        for seq in inputs:
            vocab_freq_counter.update(seq)
            
    vocab_frequencies = dict(vocab_freq_counter)
    print(f"Found {len(vocab_frequencies)} unique tokens in training set.")
except FileNotFoundError:
    print("Không tìm thấy train_10k.csv, sẽ dùng dummy frequencies.")
    vocab_frequencies = {1: 1000} # Dummy

## 4. Kiểm thử sinh văn bản với GTF-Penalty

In [ ]:
def process_qwen(text):
    text = str(text)
    idx = text.find('\n\nTóm tắt:')
    if idx == -1: idx = text.find('\nTóm tắt:')
    if idx == -1: idx = text.find('Tóm tắt bài báo sau:')
    if idx != -1: text = text[:idx].strip()
    for token in ['<|endoftext|>', '<|im_end|>', '<|im_start|>']:
        text = text.replace(token, '')
    return text.strip()

def generate_summary(article_text, use_gtf=True):
    prompt = f"Tóm tắt bài báo sau:\n{article_text}\n\nTóm tắt:\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1500).to(model.device)
    input_len = inputs.input_ids.shape[1]
    
    gen_kwargs = {
        "input_ids": inputs.input_ids,
        "attention_mask": inputs.attention_mask,
        "max_new_tokens": 128,
        "num_beams": 4,
        "early_stopping": True,
        "pad_token_id": tokenizer.eos_token_id
    }
    
    if use_gtf:
        # Phải truyền prompt_length vào để tránh phạt những từ có trong bài báo gốc!
        gtf_processor = GTFPenaltyLogitsProcessor(vocab_frequencies, prompt_length=input_len, alpha=1.0, base_penalty=0.6)
        gen_kwargs["logits_processor"] = LogitsProcessorList([gtf_processor])
        
    with torch.no_grad(): outputs = model.generate(**gen_kwargs)
    summary_ids = outputs[0][input_len:]
    raw_summary = tokenizer.decode(summary_ids, skip_special_tokens=True)
    return process_qwen(raw_summary)

# Chạy thử nghiệm trên 1 bài báo mẫu
sample_article = """
Chiều 15/8, Bộ Y tế ghi nhận thêm 2 ca mắc mới Covid-19, nâng tổng số ca mắc tại Việt Nam lên 930.
Ca bệnh 929 (BN929) là bệnh nhân nữ, 57 tuổi, trú tại xã Tiên Thuận, huyện Bến Cầu, tỉnh Tây Ninh.
Bệnh nhân từ Mỹ nhập cảnh Sân bay Tân Sơn Nhất ngày 02/8 trên chuyến bay VN2, được cách ly tập trung ngay tại TP. HCM.
Sau hai lần xét nghiệm, kết quả dương tính với virus SARS-CoV-2.
"""

print("1. KHÔNG DÙNG GTF-Penalty:")
print(generate_summary(sample_article, use_gtf=False))
print("-" * 50)
print("2. CÓ DÙNG GTF-Penalty:")
print(generate_summary(sample_article, use_gtf=True))


## 5. Đánh giá toàn bộ tập Test (Tùy chọn)
Chạy vòng lặp trên `test_df` để tính lại điểm ROUGE.

In [ ]:
import evaluate
import numpy as np
from tqdm.auto import tqdm
rouge = evaluate.load('rouge')
bleu = evaluate.load('bleu')
bertscore = evaluate.load('bertscore')

try:
    test_df = pd.read_csv('test_1k.csv')
    predictions = []
    references = test_df['abstract'].tolist()
    articles = test_df['article'].tolist()
    
    print('Generating summaries for test set...')
    for article in tqdm(articles):
        pred = generate_summary(article, use_gtf=True)
        predictions.append(pred)
        
    print('Computing ROUGE...')
    rouge_results = rouge.compute(predictions=predictions, references=references)
    print('ROUGE:', rouge_results)
    
    print('Computing BLEU...')
    bleu_results = bleu.compute(predictions=predictions, references=references)
    print('BLEU:', bleu_results['bleu'] * 100)
    
    print('Computing BERTScore...')
    bert_results = bertscore.compute(predictions=predictions, references=references, lang='vi')
    print('BERTScore-F1:', np.mean(bert_results['f1']) * 100)
    import pandas as pd
    df_save = pd.DataFrame({'article': articles, 'reference': references, 'prediction': predictions})
    df_save.to_csv('qwen_gtf_predictions.csv', index=False)
    print('Đã lưu kết quả ra file qwen_gtf_predictions.csv để dùng sau!')
except FileNotFoundError:
    print('Không tìm thấy file test, bỏ qua bước đánh giá toàn tập.')
